### Needed for Login

In [ ]:
# !pip install earthscope-cli
# !pip install boto3
# !pip install earthscope-sdk
# !pip install obspy

In [ ]:
!curl ipinfo.io

In [ ]:
!es login

In [ ]:
import boto3
from botocore.config import Config
from earthscope_sdk import EarthScopeClient
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
client = EarthScopeClient()

profile = client.user.get_profile()
print(profile)

In [ ]:
creds = client.user.get_aws_credentials()
print("Credentials retrieved successfully.")

In [ ]:
# Initialize S3 Client
session = boto3.Session(
    aws_access_key_id=creds.aws_access_key_id,
    aws_secret_access_key=creds.aws_secret_access_key,
    aws_session_token=creds.aws_session_token,
)

s3_client = session.client("s3",config=Config(response_checksum_validation='when_required'))

In [ ]:
S3_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
BUCKET = S3_ACCESS_POINT
PREFIX = "miniseed/"

### Check What Networks are available

In [ ]:
list_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")
nets = [c["Prefix"].split("/", 1)[1] for c in list_resp["CommonPrefixes"]]
print(nets)

In [ ]:
get_resp = s3_client.get_object(
    Bucket=BUCKET,
    #Key=f"{PREFIX}IU/2024/300/MBW.UW.2024.300#2",  # ~6 MB
    # Key=f"{PREFIX}UW/2024/300/MPO.UW.2024.300#2",  # ~85 MB
    Key=f"{PREFIX}UW/2024/300/SLA.UW.2024.300#2",  # ~400 MB
)

In [ ]:
print(dir(s3_client))

### List of All Data Available for a Particular Network

In [ ]:
## Example to browse through the cloud
network = 'IM'
network_prefix = f"miniseed/{network}/2024/065/"
list_resp_years = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=network_prefix)

years = []
years = [c["Key"].split('/') for c in list_resp_years["Contents"]]

print(years)
len(list_resp_years["Contents"])

In [ ]:
print(nets)

In [ ]:
## Example to Prepare a Sample Data Availability List in Cloud for a Network for Few Years
network = 'IM'
datalist = []
list_of_years = np.arange(2003,2005,1)
list_of_days = [f'{i:03d}' for i in range(1, 367)]

for years in list_of_years:
    for days in list_of_days:
      data_prefix = f"miniseed/{network}/{years}/{days}/"
      try:
        data_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=data_prefix)
        for c in data_resp["Contents"]:
            dataacess_key = c["Key"]
            year = dataacess_key.split('/')[2]
            yearday = dataacess_key.split('/')[3]
            station = dataacess_key.split('/')[-1].split('.')[0]
            datalist.append([network,station,year,yearday,dataacess_key])
      except Exception as e:
        print(e)
        continue

datalist_df = pd.DataFrame(datalist,columns=['network','station','year','yearday','dataacess_key'])
datalist_df.sort_values(by=['network','station','year','yearday'], inplace=True)
datalist_df


In [ ]:
#Same as above but with all years for one station
network = '1B'
datalist = []

data_prefix = f"miniseed/{network}/"

# Create a reusable paginator
paginator = s3_client.get_paginator('list_objects_v2')

try:
    pages = paginator.paginate(Bucket=BUCKET, Prefix=data_prefix)

    for page in pages:
        if "Contents" not in page:
            continue

        for c in page["Contents"]:
            dataacess_key = c["Key"]

            parts = dataacess_key.split('/')

            if len(parts) >= 5:
                extracted_year = parts[2]
                yearday = parts[3]
                station = parts[-1].split('.')[0]

                datalist.append([network, station, extracted_year, yearday, dataacess_key])

except Exception as e:
    print(f"Error accessing prefix {data_prefix}: {e}")

datalist_df = pd.DataFrame(datalist, columns=['network', 'station', 'year', 'yearday', 'dataacess_key'])
datalist_df.sort_values(by=['network', 'station', 'year', 'yearday'], inplace=True)

if not datalist_df.empty:
    discovered_years = datalist_df['year'].unique()
    print(f"Success! Found {len(datalist_df)} files for network {network}.")
    print(f"Years found in bucket: {discovered_years}")
else:
    print(f"DataFrame is still empty. Network '{network}' does not exist in this S3 bucket.")

datalist_df

In [ ]:
len(set(datalist_df['station']))

In [ ]:
!ls

In [ ]:
(df_coords["network"] =='1B').sum()

In [ ]:
df_coords[df_coords["network"] == '1B']['station'].nunique()

In [ ]:
# 1. Get the unique stations from your IRIS metadata
meta_stations = set(df_coords[df_coords['network'] == '1B']['station'])

# 2. Get the unique stations actually sitting in your S3 bucket
data_stations = set(datalist_df['station'])

# 3. Find stations that are in the metadata, but missing from S3
missing_from_s3 = meta_stations - data_stations

# 4. Find stations that are in S3, but missing from metadata (rare, but happens!)
missing_from_meta = data_stations - meta_stations

print(f"Stations in IRIS but missing from S3 data: {len(missing_from_s3)}")
if missing_from_s3:
    print(missing_from_s3)

print(f"\nStations in S3 data but missing from IRIS metadata: {len(missing_from_meta)}")
if missing_from_meta:
    print(missing_from_meta)

In [ ]:
datalist_df.groupby(['network','station','year']).count().reset_index()

In [ ]:
datalist_df[datalist_df['station'].str.contains('ATTU')]

### Download Data

In [ ]:
import os
from datetime import datetime

def download_daily_data(s3_client, bucket_name, net, prefix, output_dir='/content/drive/MyDrive/PRJ_Wavenet_Terraseis/sample_data'):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Starting download for {prefix}")

    try:
        local_path = os.path.join(output_dir, net)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)

        #s3_client.download_file(bucket_name, prefix, local_path)
        s3_client.get_object(Bucket=bucket_name, Key=prefix)
        print("Received the Object")

    except Exception as e:
        print(f"Error accessing network {net}: {e}")
        print("Download process not completed.")

In [ ]:
# Configuration for 2004 Sumatra Earthquake
S3_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
BUCKET = S3_ACCESS_POINT

sumatra_date = '2004-12-26'
date_obj = datetime.strptime(sumatra_date, '%Y-%m-%d')
yearday = date_obj.strftime('%j')
year = date_obj.strftime('%Y')


network='UW'
filtered_df = datalist_df[(datalist_df['station'].str.contains('ATTU')) & (datalist_df['year']==year) & (datalist_df['yearday']==yearday)]
print(filtered_df['dataacess_key'].item())

## Run the download
#download_daily_data(s3_client, BUCKET, net='IM', prefix="miniseed/UW/2024/300/MBW.UW.2024.300#2")

s3_client.get_object(Bucket=BUCKET, Key="miniseed/UW/2024/300/MPO.UW.2024.300#2")

In [ ]:
s3_client = session.client("s3")
get_resp = s3_client.get_object(
    Bucket=BUCKET,
    Key=f"{PREFIX}UW/2024/300/MBW.UW.2024.300#2",  # ~6 MB
    # Key=f"{PREFIX}UW/2024/300/MPO.UW.2024.300#2",  # ~85 MB
    # Key=f"{PREFIX}UW/2024/300/SLA.UW.2024.300#2",  # ~400 MB
)

### Draft Codes

In [ ]:
# network = 'IM'
# datalist = []
# network_prefix = f"miniseed/{network}/"
# resp_years = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=network_prefix, Delimiter='/')

# if 'CommonPrefixes' in resp_years:
#     # Extract year strings from the prefixes (e.g., 'miniseed/IM/1998/' -> '1998')
#     available_years = [cp['Prefix'].split('/')[-2] for cp in resp_years['CommonPrefixes']]

#     for year in available_years:
#         # 2. For each available year, list available days
#         year_prefix = f"miniseed/{network}/{year}/"
#         resp_days = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=year_prefix, Delimiter='/')

#         if 'CommonPrefixes' in resp_days:
#             available_days = [cp['Prefix'].split('/')[-2] for cp in resp_days['CommonPrefixes']]

#             for day in available_days:
#                 # 3. List actual files in the specific day directory
#                 day_prefix = f"miniseed/{network}/{year}/{day}/"
#                 resp_files = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=day_prefix)

#                 if 'Contents' in resp_files:
#                     for c in resp_files['Contents']:
#                         dataacess_key = c['Key']
#                         # Parse filename: usually NET.STA.YEAR.JDAY...
#                         # Safely extracting parts
#                         parts = dataacess_key.split('/')
#                         # parts usually: ['miniseed', 'IM', '1998', '108', 'filename']

#                         station = parts[-1].split('.')[0]

#                         datalist.append([network, station, year, day, dataacess_key])

# datalist_df = pd.DataFrame(datalist, columns=['network', 'station', 'year', 'yearday', 'dataacess_key'])
# datalist_df.sort_values(by=['network', 'station', 'year', 'yearday'], inplace=True)

In [ ]:
# import os
# from datetime import datetime

# def download_daily_data(s3_client, bucket_name, date_str, networks, output_dir='data'):
#     """
#     Downloads data for specific networks on a specific day.

#     Args:
#     s3_client: Boto3 S3 client object.
#     bucket_name: Name of the S3 bucket.
#     date_str: Date string in 'YYYY-MM-DD' format.
#     networks: List of network codes (e.g., ['IU', 'II']).
#     output_dir: Local directory to save files.
#     """
#     target_date = datetime.strptime(date_str, '%Y-%m-%d')
#     year = target_date.strftime('%Y')
#     jday = target_date.strftime('%j')

#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)

#     print(f"Starting download for {date_str} (Year: {year}, JDay: {jday})...")

#     for net in networks:
#         # constructing a likely prefix.
#         # Note: Adjust this structure based on the actual S3 bucket layout.
#         # Common layout: miniseed/YEAR/NET/STA/...
#         prefix = f"miniseed/{year}/"

#         print(f"Scanning network: {net} with prefix: {prefix}")

#         try:
#             paginator = s3_client.get_paginator('list_objects_v2')
#             page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

#             file_count = 0
#             for page in page_iterator:
#                 if 'Contents' in page:
#                     for obj in page['Contents']:
#                         key = obj['Key']

#                         # Filter by day if the filename contains day information
#                         # Assuming filename format like: NET.STA.LOC.CHAN.D.YEAR.JDAY
#                         if f".{year}.{jday}" in key:
#                             filename = os.path.basename(key)
#                             local_path = os.path.join(output_dir, net, filename)

#                             # Create network subdirectory
#                             os.makedirs(os.path.dirname(local_path), exist_ok=True)

#                             print(f"Downloading {key} -> {local_path}")
#                             s3_client.download_file(bucket_name, key, local_path)
#                             file_count += 1

#             if file_count == 0:
#                 print(f"No files found for network {net} on {date_str} with prefix {prefix}")

#         except Exception as e:
#             print(f"Error accessing network {net}: {e}")

#     print("Download process completed.")

In [ ]:
# import boto3
# import os
# from earthscope_sdk import EarthScopeClient

# # 1. Initialize EarthScope Client and get temporary AWS Credentials
# print("Authenticating with EarthScope...")
# client = EarthScopeClient()
# creds = client.user.get_aws_credentials()

# # 2. Create a boto3 Session with the temporary credentials
# session = boto3.Session(
#     aws_access_key_id=creds.aws_access_key_id,
#     aws_secret_access_key=creds.aws_secret_access_key,
#     aws_session_token=creds.aws_session_token,
# )
# s3_client = session.client("s3")

# # 3. Define the S3 Access Point and target data
# S3_ACCESS_POINT = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
# BUCKET = S3_ACCESS_POINT

# # Define the specific day you want to download
# NETWORK = "UW"
# YEAR = "2024"
# DAY_OF_YEAR = "300" # This is the Julian day of the year

# # The prefix structure is miniseed/NETWORK/YEAR/DAY_OF_YEAR/
# target_prefix = f"miniseed/{NETWORK}/{YEAR}/{DAY_OF_YEAR}/"

# # 4. Prepare local directory for downloads
# output_dir = f"data_{NETWORK}_{YEAR}_{DAY_OF_YEAR}"
# os.makedirs(output_dir, exist_ok=True)

# # 5. List and download the data
# print(f"Searching for data in: {target_prefix}")

# # We use a paginator in case there are more than 1000 files in this day/network
# paginator = s3_client.get_paginator('list_objects_v2')
# pages = paginator.paginate(Bucket=BUCKET, Prefix=target_prefix)

# download_count = 0
# total_bytes = 0

# for page in pages:
#     if "Contents" in page:
#         for obj in page["Contents"]:
#             key = obj["Key"]
#             file_size = obj["Size"]

#             # Extract the actual filename from the end of the S3 key path
#             filename = key.split("/")[-1]
#             download_path = os.path.join(output_dir, filename)

#             print(f"Downloading {filename} ({(file_size / 1024 / 1024):.2f} MB)...")

#             # Download the file to the local directory
#             try:
#                 s3_client.download_file(BUCKET, key, download_path)
#                 download_count += 1
#                 total_bytes += file_size
#             except Exception as e:
#                 print(f"Failed to download {filename}. Error: {e}")
#                 # This might happen if you hit a restricted file you don't have access to

# print("-" * 30)
# print(f"Finished! Successfully downloaded {download_count} files.")
# print(f"Total data downloaded: {(total_bytes / 1024 / 1024):.2f} MB.")
# print(f"Files saved to: ./{output_dir}/")